# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and process the FAIR² dataset package on protein abundance in human extracellular vesicles using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets and their fields (`@id`), using the Croissant schema.

In [ ]:
# Helper function to print record sets and their fields and columns by @id
def print_recordsets_overview(ds):
    print("Record Sets in this dataset:")
    for rs in ds.metadata.record_sets:
        print(f"  RecordSet name: {rs.name}, @id: {rs.id}")
        print("    Fields:")
        for f in rs.fields:
            print(f"      - {f.name} (@id: {f.id}) [dataType: {f.data_type}] column: {f.column}")
        print("    Columns:")
        for c in rs.columns:
            print(f"      - {c.name} (@id: {c.id})")
        print()

print_recordsets_overview(dataset)

## 3. Data Extraction
Load data from each record set into DataFrames for analysis using their `@id`s.

In [ ]:
# List all record sets by @id
record_sets = [rs.id for rs in dataset.metadata.record_sets]
print(f"RecordSet IDs: {record_sets}")

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet {record_set_id} with shape {df.shape}")
    else:
        print(f"WARNING: No records found for RecordSet {record_set_id}")

# For demonstration, pick the first record set for EDA
if len(record_sets) > 0:
    default_rs_id = record_sets[0]
    print(f"Available fields/columns: {dataframes[default_rs_id].columns.tolist() if default_rs_id in dataframes else []}")
    if default_rs_id in dataframes:
        display(dataframes[default_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Clean, filter, and transform the data using field `@id`s. Adjust field ids as appropriate for your analysis below.

In [ ]:
from IPython.display import display

# Set which record set to use, and which numeric field (by @id)
record_set_id = default_rs_id  # You can change this to another RecordSet @id if desired
df = dataframes[record_set_id]

# Pick a numeric field for analysis. List all columns so user can choose.
print("Columns in DataFrame:", df.columns.tolist())

# Try fields likely to be numeric (modify as appropriate to your dataset)
possible_numeric_fields = [col for col in df.columns if ('coverage' in col.lower() or 'count' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower() or 'pi' in col.lower() or df[col].dtype in [int, float])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Using numeric field '{numeric_field}' (@id) for EDA.")
else:
    raise ValueError("No plausible numeric field found for EDA.")

# Drop NA values for this numeric analysis
filtered_df = df.dropna(subset=[numeric_field])

# Example threshold, adjust to a value that makes sense for your data
threshold = filtered_df[numeric_field].quantile(0.75)
filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold:.2f} (75th percentile):")
display(filtered_df.head())

# Normalize numeric field (zero mean, unit std)
filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized values for '{numeric_field}':")
display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Find a group field (string/categorical)
possible_group_fields = [col for col in df.columns if (df[col].dtype == 'O' and col != numeric_field)]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"Grouping by field '{group_field}' (@id).")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    display(grouped_df.head())
else:
    print("No suitable grouping field found for EDA.")

## 5. Visualization
Visualize distributions or relationships using the chosen fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If grouped, show boxplot
if 'group_field' in locals() and group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we have loaded, explored, and visualized the FAIR² mass spec dataset using `mlcroissant`. The data reveal distribution patterns and summary statistics on protein abundance and related attributes. For deeper biological or domain-specific insights, refer to the dataset documentation and study further with the extracted record sets.

*To adapt this notebook for another dataset, update the Croissant schema URL and re-run the notebook. Use field and record set `@id`s shown above to select your analysis target fields and tables. All dataset references use the `@id` as required by best practices for Croissant datasets.*